# GraphFrames and the Panama Papers  
## Graph algorithms for investigative network analysis

In the previous class, we built a Panama Papers graph from officers, entities, intermediaries, addresses, and relationships. In this class, we move from descriptive analysis to graph algorithms.

We will use GraphFrames to explore PageRank, breadth-first search, shortest paths, strongly connected components, label propagation, and triangle count.

Graph algorithms identify patterns, hubs, communities, and paths. They do not prove misconduct. The goal is to learn how these algorithms help us inspect a large network more intelligently.


## 0. Lightning.ai / GraphFrames setup

Run the following setup cells first. These are the same preamble cells used in Notebook 1.

In [1]:
#Checking the installed Java version

!java -version

!pip install "pyspark==3.5.0" 

# Install Java 17

!sudo apt-get update

!sudo apt-get install -y openjdk-17-jdk-headless

!java -version

%pip install graphframes-py==0.10.0

# Set JAVA_HOME to Java 17

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GraphFramesWithSpark4") \
    .config("spark.jars.packages", "io.graphframes:graphframes-spark3_2.12:0.10.0") \
    .getOrCreate()

print(f"spark version: {spark.version}")

print("spark session created with graphframes package specified!")

#import the package we just installed

from graphframes import *

#import data types - All data types of Spark SQL are located in the package of pyspark.sql.types

from pyspark.sql.types import *

#row can be used to create a row object by using named arguments

from pyspark.sql import Row

from pyspark.sql.functions import col

sc = spark.sparkContext

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-124.04.1, mixed mode, sharing)
Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://download.docker.com/linux/ubuntu noble InRelease                 
Get:3 https://cli.github.com/packages stable InRelease [3917 B]                
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://cloud.archive.ubuntu.com/ubuntu noble InRelease                  
Hit:8 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease          
Hit:9 https://security.ubuntu.com/ubuntu noble-security InRelease              
Hit:10 https://cloud.archive.ubuntu.com/ubuntu noble-backports InRelease       
Hit:11 https://archive.ubuntu.com/ubu

Ivy Default Cache set to: /home/zeus/.ivy2/cache
The jars for the packages stored in: /home/zeus/.ivy2/jars
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-716cb9b9-cfc8-4893-8042-bc1368e485d9;1.0
	confs: [default]
	found io.graphframes#graphframes-spark3_2.12;0.10.0 in central
	found io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 in central
:: resolution report :: resolve 152ms :: artifacts dl 10ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 from central in [default]
	io.graphframes#graphframes-spark3_2.12;0.10.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   || 

spark version: 3.5.0
spark session created with graphframes package specified!


## 1. Imports and configuration

This notebook intentionally uses a few **teaching-size graphs** for expensive algorithms.  
You can increase the limits if your Lightning.ai instance has enough memory.

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from graphframes import GraphFrame

# GraphFrames connected components and strongly connected components need checkpointing.
spark.sparkContext.setCheckpointDir("/tmp/graphframes-checkpoints")



# Keep this False for teaching sessions.
# Set to True only if your environment can handle larger jobs.
RUN_FULL_GRAPH_ALGORITHMS = False

# Limits used to create manageable teaching graphs.
MAX_EDGES_FOR_ALGORITHMS = 200_000
MAX_SHARED_FEATURES_FOR_PROJECTION = 2_000
MAX_ENTITY_PAIRS_PER_FEATURE = 100_000

spark.conf.set("spark.sql.shuffle.partitions", "64")

## 2. Load the same CSV files

We reuse the same five CSVs:

- `nodes_addresses.csv`
- `nodes_entities.csv`
- `nodes_intermediaries.csv`
- `nodes_officers.csv`
- `relationships.csv`

The important GraphFrames requirement is:

- vertices must have a column called `id`
- edges must have columns called `src` and `dst`

In [3]:
# Change this path if your files are stored elsewhere
DATA_DIR = "/teamspace/studios/this_studio/week10/data"

addresses_path = f"{DATA_DIR}/nodes-addresses.csv"
entities_path = f"{DATA_DIR}/nodes-entities.csv"
intermediaries_path = f"{DATA_DIR}/nodes-intermediaries.csv"
officers_path = f"{DATA_DIR}/nodes-officers.csv"
relationships_path = f"{DATA_DIR}/relationships.csv"

addresses_raw = spark.read.csv(addresses_path, header=True, inferSchema=True, multiLine=True, escape='"')
entities_raw = spark.read.csv(entities_path, header=True, inferSchema=True, multiLine=True, escape='"')
intermediaries_raw = spark.read.csv(intermediaries_path, header=True, inferSchema=True, multiLine=True, escape='"')
officers_raw = spark.read.csv(officers_path, header=True, inferSchema=True, multiLine=True, escape='"')
relationships_raw = spark.read.csv(relationships_path, header=True, inferSchema=True, multiLine=True, escape='"')

print("Loaded files.")
print("Addresses:", addresses_raw.count())
print("Entities:", entities_raw.count())
print("Intermediaries:", intermediaries_raw.count())
print("Officers:", officers_raw.count())
print("Relationships:", relationships_raw.count())
print("CSV files loaded.")
print("addresses:", addresses_raw.count())
print("entities:", entities_raw.count())
print("intermediaries:", intermediaries_raw.count())
print("officers:", officers_raw.count())
print("relationships:", relationships_raw.count())

Loaded files.


Addresses: 402246


Entities: 814344
Intermediaries: 25629
Officers: 771315


Relationships: 3339267
CSV files loaded.
addresses: 402246


entities: 814344
intermediaries: 25629
officers: 771315


relationships: 3339267


## 3. Build vertices

We create one unified vertex DataFrame.  
The column `node_type` is crucial because this is a **heterogeneous graph**.

In [4]:
def standardize_vertices(df, node_type, name_expr):
    available_cols = df.columns

    base = (
        df
        .withColumn("id", F.col("node_id").cast("string"))
        .withColumn("node_type", F.lit(node_type))
        .withColumn("display_name", name_expr)
    )

    # Keep useful columns if they exist; create nulls otherwise.
    wanted = ["id", "display_name", "node_type", "name", "address", "countries",
              "country_codes", "jurisdiction", "jurisdiction_description",
              "sourceID", "status", "service_provider", "valid_until"]

    for c in wanted:
        if c not in base.columns:
            base = base.withColumn(c, F.lit(None).cast("string"))

    return base.select(*wanted)

addresses_v = standardize_vertices(
    addresses_raw,
    "address",
    F.coalesce(F.col("name"), F.col("address"), F.col("node_id"))
)

entities_v = standardize_vertices(
    entities_raw,
    "entity",
    F.coalesce(F.col("name"), F.col("original_name"), F.col("node_id"))
)

intermediaries_v = standardize_vertices(
    intermediaries_raw,
    "intermediary",
    F.coalesce(F.col("name"), F.col("node_id"))
)

officers_v = standardize_vertices(
    officers_raw,
    "officer",
    F.coalesce(F.col("name"), F.col("node_id"))
)

vertices = (
    addresses_v
    .unionByName(entities_v, allowMissingColumns=True)
    .unionByName(intermediaries_v, allowMissingColumns=True)
    .unionByName(officers_v, allowMissingColumns=True)
    .dropDuplicates(["id"])
    .cache()
)

vertices.groupBy("node_type").count().orderBy("node_type").show()

+------------+------+
|   node_type| count|
+------------+------+
|     address|402246|
|      entity|814344|
|intermediary| 25629|
|     officer|771315|
+------------+------+



## 4. Build edges

We rename the relationship endpoints to GraphFrames' required names: `src` and `dst`.

We also preserve:

- `rel_type`
- `link`
- `sourceID`
- `status`
- `start_date`
- `end_date`

In [5]:
edges = (
    relationships_raw
    .withColumn("src", F.col("node_id_start").cast("string"))
    .withColumn("dst", F.col("node_id_end").cast("string"))
    .withColumn("rel_type_clean", F.lower(F.trim(F.col("rel_type"))))
    .select(
        "src", "dst",
        "rel_type",
        "rel_type_clean",
        "link",
        "status",
        "start_date",
        "end_date",
        "sourceID"
    )
    .dropna(subset=["src", "dst"])
    .cache()
)

edges.groupBy("rel_type_clean").count().orderBy(F.desc("count")).show(30, truncate=False)

+------------------------+-------+
|rel_type_clean          |count  |
+------------------------+-------+
|officer_of              |1720357|
|registered_address      |832721 |
|intermediary_of         |598546 |
|same_name_as            |104170 |
|similar                 |46761  |
|same_company_as         |15523  |
|connected_to            |12145  |
|same_as                 |4272   |
|same_id_as              |3120   |
|underlying              |1308   |
|similar_company_as      |203    |
|probably_same_officer_as|132    |
|same_address_as         |5      |
|same_intermediary_as    |4      |
+------------------------+-------+



## 5. Create the full GraphFrame

This is the full heterogeneous Panama Papers graph.

In [6]:
g = GraphFrame(vertices, edges)

print("vertices:", g.vertices.count())
print("edges:", g.edges.count())

vertices: 2013534
edges: 3339267


## PageRank

### What is PageRank?

PageRank was originally invented by Larry Page and Sergey Brin (the founders of Google) in 1998 to rank web pages in search results. The core idea is simple but powerful: **a node is important if it is connected to other important nodes.**

This creates a circular definition. PageRank solves it iteratively.

### How does it work?

Imagine a person randomly walking through the graph. At each step, they follow a random outgoing edge to a new node. Occasionally (with probability `resetProbability`, set to 0.15 by default), they get bored and jump to a completely random node in the graph instead of following an edge.

After doing this for a very long time, the fraction of time spent at each node is its PageRank score. Nodes that are visited more often are more "important."

Concretely, the algorithm works in rounds:

1. Every node starts with the same score (1/N, where N is the total number of nodes)
2. In each iteration, every node distributes its current score equally across all its outgoing edges
3. Each node's new score is the sum of what it received from its incoming edges, plus a small base amount from the random jump factor
4. Repeat for `maxIter` iterations (we use 5)

### How is it different from degree?

**Degree** simply counts how many edges touch a node. A node with 100 connections always has degree 100, regardless of whether those connections come from isolated nodes or from major hubs.

**PageRank** accounts for the quality of connections. Consider two entities, both with 10 officers:
- Entity A's 10 officers each serve on only 1 board (Entity A). These officers have low PageRank themselves, so they pass little importance to Entity A.
- Entity B's 10 officers each serve on 50 boards across the network. These officers have high PageRank, so Entity B inherits more importance.

Both have degree 10, but Entity B will have a higher PageRank.

### Why does it matter for the Panama Papers?

In this network, edges are directed — "shareholder of", "registered address", "intermediary of" all point from one node to another. PageRank respects this direction, which means:
- **Addresses and entities** tend to *receive* links (officers point to them, intermediaries point to them), so they accumulate PageRank
- **Officers and intermediaries** tend to *send* links outward, so their PageRank stays lower unless they themselves receive links from important nodes

We should always interpret PageRank results **by node type**, because a high-ranking address and a high-ranking officer mean very different things structurally.

### Parameters we use

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `resetProbability` | 0.15 | 15% chance of jumping to a random node at each step (the classic "damping factor" is 1 − 0.15 = 0.85) |
| `maxIter` | 5 | Number of iterations — usually 5–20 is sufficient for convergence on large graphs |

In [7]:
pagerank_results = g.pageRank(resetProbability=0.15, maxIter=5)



26/05/17 19:51:14 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.
26/05/17 19:51:14 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.
26/05/17 19:51:15 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.
26/05/17 19:51:15 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.
26/05/17 19:51:16 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.
26/05/17 19:51:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/17 19:51:27 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.
26/05/17 19:51:27 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.
26/05/17 19:51:27 WA

In [8]:
degrees_df   = g.degrees.withColumnRenamed("id", "deg_id")
in_deg_df    = g.inDegrees.withColumnRenamed("id", "in_id")
out_deg_df   = g.outDegrees.withColumnRenamed("id", "out_id")

pagerank_vertices = (
    pagerank_results.vertices
    .select("id", "display_name", "node_type", "countries", "jurisdiction",
            "jurisdiction_description", "status", "service_provider",
            "sourceID", "pagerank")
    .join(degrees_df,  F.col("id") == F.col("deg_id"), how="left").drop("deg_id")
    .join(in_deg_df,   F.col("id") == F.col("in_id"),  how="left").drop("in_id")
    .join(out_deg_df,  F.col("id") == F.col("out_id"), how="left").drop("out_id")
    .cache()
)
pagerank_vertices.orderBy(F.desc("pagerank")).show(20, truncate=80)

26/05/17 19:51:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 19:51:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 19:51:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 19:51:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 19:51:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 19:51:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+---------+--------------------------------------------------------------------------------+---------+--------------+------------+------------------------+---------+----------------+---------------------------------------------+------------------+------+--------+---------+
|       id|                                                                    display_name|node_type|     countries|jurisdiction|jurisdiction_description|   status|service_provider|                                     sourceID|          pagerank|degree|inDegree|outDegree|
+---------+--------------------------------------------------------------------------------+---------+--------------+------------+------------------------+---------+----------------+---------------------------------------------+------------------+------+--------+---------+
| 81027146|            Clifton House; 75 Fort Street; Grand Cayman KY1-1108; Cayman Islands|  address|Cayman Islands|        NULL|                    NULL|     NULL|            N

The top PageRank table should be read together with `node_type`, `degree`, `inDegree`, and `outDegree`.

If top PageRank nodes are addresses or entities, PageRank is mostly identifying nodes that receive many important links. If top PageRank nodes are intermediaries or officers, it may indicate repeated links outward to influential parts of the graph.


In [9]:
window_by_type = Window.partitionBy("node_type").orderBy(F.desc("pagerank"))

top_pagerank_by_type = (
    pagerank_vertices
    .withColumn("rank_within_type", F.row_number().over(window_by_type))
    .filter(F.col("rank_within_type") <= 10)
    .select("node_type", "rank_within_type", "id", "display_name",
            "countries", "jurisdiction", "status", "service_provider",
            "degree", "inDegree", "outDegree", "pagerank")
    .orderBy("node_type", "rank_within_type")
)

top_pagerank_by_type.show(80, truncate=80)


+------------+----------------+---------+--------------------------------------------------------------------------------+----------------------+------------+-------------+----------------+------+--------+---------+------------------+
|   node_type|rank_within_type|       id|                                                                    display_name|             countries|jurisdiction|       status|service_provider|degree|inDegree|outDegree|          pagerank|
+------------+----------------+---------+--------------------------------------------------------------------------------+----------------------+------------+-------------+----------------+------+--------+---------+------------------+
|     address|               1| 81027146|            Clifton House; 75 Fort Street; Grand Cayman KY1-1108; Cayman Islands|        Cayman Islands|        NULL|         NULL|            NULL|  9268|    9268|     NULL|1718.7207248914374|
|     address|               2|   285729|Sealight Incorporat

Ranking within node type is usually more useful than one global ranking.

This lets us compare addresses with addresses, intermediaries with intermediaries, officers with officers, and entities with entities. It also helps avoid the mistake of treating every node type as if it means the same thing.


In [10]:
# PageRank by country and node type.
# This gives a higher-level view without losing the node-type distinction.

pagerank_country_summary = (
    pagerank_vertices
    .filter(F.col("countries").isNotNull())
    .withColumn("country", F.explode(F.split(F.col("countries"), ";")))
    .withColumn("country", F.trim(F.col("country")))
    .filter(F.col("country") != "")
    .groupBy("country", "node_type")
    .agg(
        F.count("*").alias("num_nodes"),
        F.round(F.avg("pagerank"), 6).alias("avg_pagerank"),
        F.round(F.max("pagerank"), 6).alias("max_pagerank"),
        F.max("degree").alias("max_degree")
    )
    .filter(F.col("num_nodes") >= 5)
    .orderBy(F.desc("max_pagerank"))
)

pagerank_country_summary.show(30, truncate=False)


+----------------------+---------+---------+------------+------------+----------+
|country               |node_type|num_nodes|avg_pagerank|max_pagerank|max_degree|
+----------------------+---------+---------+------------+------------+----------+
|Cayman Islands        |address  |459      |5.070383    |1718.720725 |9268      |
|Hong Kong             |address  |5541     |1.210419    |590.377542  |3896      |
|Aruba                 |address  |3505     |1.380756    |372.377231  |2339      |
|Samoa                 |address  |222      |2.677707    |225.041178  |1615      |
|SYC                   |address  |6        |30.828129   |177.918217  |851       |
|Hong Kong             |entity   |10420    |1.075728    |137.298718  |1007      |
|Russia                |address  |1496     |1.2339      |100.835529  |557       |
|Barbados              |address  |3812     |1.379115    |76.633176   |304       |
|Vanuatu               |address  |9        |8.816614    |71.563612   |438       |
|Monaco         

This table uses the schema rather than only the graph structure.

A country can have many low-ranking nodes, or only a few nodes with one major hub. The `max_pagerank` and `max_degree` columns help identify where the strongest individual nodes are located.


In [11]:
# Compare PageRank and degree.
# This helps show whether PageRank is simply reproducing degree or adding something different.

pagerank_vs_degree = (
    pagerank_vertices
    .select("id", "display_name", "node_type", "countries",
            "degree", "inDegree", "outDegree", "pagerank")
    .orderBy(F.desc("pagerank"))
)

pagerank_vs_degree.show(30, truncate=80)


+---------+--------------------------------------------------------------------------------+---------+--------------+------+--------+---------+------------------+
|       id|                                                                    display_name|node_type|     countries|degree|inDegree|outDegree|          pagerank|
+---------+--------------------------------------------------------------------------------+---------+--------------+------+--------+---------+------------------+
| 81027146|            Clifton House; 75 Fort Street; Grand Cayman KY1-1108; Cayman Islands|  address|Cayman Islands|  9268|    9268|     NULL|1718.7207248914374|
|   285729|Sealight Incorporations Limited Room 1201, Connaught Commercial Building 185 ...|  address|     Hong Kong|  3896|    3896|     NULL| 590.3775422902185|
| 58007938|                                          171 OLD BAKERY STREET, VALLETTA, MALTA|  address|          NULL|  2505|    2505|     NULL| 484.3555758173762|
| 88002083|           

This comparison reveals where PageRank adds value beyond degree:
- **High PageRank, low degree** nodes are connected to a small number of very important nodes. In the Panama Papers, these might be entities linked to only a few officers — but those officers are themselves connected to hundreds of other entities across the network.
- **High degree, low PageRank** nodes have many connections, but to peripheral parts of the graph. They are locally busy but globally unimportant.

If PageRank and degree always agreed perfectly, there would be no reason to compute PageRank. The cases where they disagree are where the algorithm earns its value.

In [12]:
# Identify nodes where PageRank and degree disagree.
# These are the cases where PageRank adds information beyond simple connectivity.
# "High PageRank, low degree" = connected to a few very important nodes.
# "High degree, low PageRank" = many connections, but to unimportant nodes.

from pyspark.sql.window import Window

# Compute percentile ranks for both metrics within each node type
w = Window.partitionBy("node_type")

pagerank_degree_comparison = (
    pagerank_vertices
    .filter(F.col("degree").isNotNull() & (F.col("degree") > 0))
    .withColumn("pagerank_pctile", F.percent_rank().over(w.orderBy("pagerank")))
    .withColumn("degree_pctile",   F.percent_rank().over(w.orderBy("degree")))
    .withColumn("pctile_gap", F.round(F.col("pagerank_pctile") - F.col("degree_pctile"), 4))
)

# Nodes that punch above their weight: high PageRank relative to degree
print("Top 15 nodes where PageRank is much HIGHER than degree would predict:")
(pagerank_degree_comparison
    .filter(F.col("degree") >= 3)
    .orderBy(F.desc("pctile_gap"))
    .select("display_name", "node_type", "countries", "degree", "pagerank",
            "degree_pctile", "pagerank_pctile", "pctile_gap")
    .show(15, truncate=60)
)

# Nodes that punch below their weight: high degree but low PageRank
print("Top 15 nodes where degree is much HIGHER than PageRank would predict:")
(pagerank_degree_comparison
    .filter(F.col("degree") >= 10)
    .orderBy("pctile_gap")
    .select("display_name", "node_type", "countries", "degree", "pagerank",
            "degree_pctile", "pagerank_pctile", "pctile_gap")
    .show(15, truncate=60)
)

Top 15 nodes where PageRank is much HIGHER than degree would predict:


+---------------------------+------------+----------------------+------+------------------+-------------------+------------------+----------+
|               display_name|   node_type|             countries|degree|          pagerank|      degree_pctile|   pagerank_pctile|pctile_gap|
+---------------------------+------------+----------------------+------+------------------+-------------------+------------------+----------+
|            CAFFERATA & CO.|intermediary|               Bahamas|     3|1.6014162137014267|0.45236220472440947|0.9990157480314961|    0.5467|
|    OCEANA EQUITIES LIMITED|      entity|             Gibraltar|     3| 7.598298068817173| 0.6006325033006847|0.9995701433878842|    0.3989|
|   DIANA GROUP PARTNERS S.A|      entity|                  NULL|     3| 7.684847452303875| 0.6006325033006847|0.9995762841966287|    0.3989|
|        COOPERSTOWN LIMITED|      entity|              Guernsey|     3| 4.911833641564302| 0.6006325033006847|0.9986305996499739|     0.398|
|     

## Breadth-first search (BFS)

Breadth-first search is one of the oldest and most fundamental graph algorithms. Given a starting node, it visits all neighbours one hop away, then all neighbours two hops away, and so on, expanding outward in concentric "rings" of distance from the source. It stops either when the entire reachable graph has been explored or when a target condition is met.

### The general idea

If you imagine the graph as a city and the nodes as locations, BFS is the algorithm you'd use to answer "which places can I reach within 10 minutes of walking?" — it explores in order of distance, not in order of which path looks interesting. This matters because the first path BFS finds between two nodes is guaranteed to be the *shortest* path in terms of number of edges.

### When to use BFS

BFS is the right tool when your question involves any of the following:

- **Reachability**: "Which nodes are within k hops of node X?"
- **Shortest unweighted path**: "What is the minimum number of relationships separating A and B?"
- **Constrained traversal**: "Find paths from any node of type X to any node of type Y, but only following edges of type Z."
- **Neighbourhood expansion**: "Show me the local context around this small set of seed nodes."

In contrast, BFS is **not** the right tool when you only need to find direct (1-hop) relationships between two sets of nodes — for that, a join is simpler and faster. BFS earns its complexity when the path length is variable, or when you need to filter edges along the way.

### BFS in GraphFrames

GraphFrames exposes BFS through three main parameters:

| Parameter | What it controls |
|-----------|------------------|
| `fromExpr` | A SQL expression identifying the starting node(s) |
| `toExpr` | A SQL expression identifying the target node(s) |
| `maxPathLength` | Maximum number of hops to search |
| `edgeFilter` | Optional: restrict which edge types BFS is allowed to traverse |

The output is a DataFrame where each row represents one path found, with columns `from`, `e0`, `v1`, `e1`, ..., `to` depending on the path length.

### Applying BFS to the Panama Papers

For our investigation, BFS lets us ask questions that pure attribute filtering can't answer. A reporter doesn't usually start by asking "give me a list of all officers from Portugal" — they ask "follow the chain of relationships from anywhere in Portugal and tell me where it leads." That's a path question, and BFS is built for it.

Below we run three BFS queries that each illustrate a different use of the algorithm: targeted reachability between two country groups, multi-source aggregation toward a single node type, and neighbourhood expansion around a small seed.

In [7]:
# Build an undirected version of g for distance-style analyses
# by unioning each edge with its reverse

reverse_edges = g.edges.select(
    F.col("dst").alias("src"),
    F.col("src").alias("dst"),
    *[c for c in g.edges.columns if c not in ("src", "dst")]
)
undirected_edges = g.edges.unionByName(reverse_edges)

g_undirected = GraphFrame(g.vertices, undirected_edges)
print(f"Undirected graph: {g_undirected.vertices.count():,} vertices, "
      f"{g_undirected.edges.count():,} edges")

Undirected graph: 2,013,534 vertices, 6,678,534 edges


### BFS 1 — EU officers reaching BVI entities

A classic offshore-finance pattern is for individuals in a developed economy to set up corporate structures in a low-tax jurisdiction. The British Virgin Islands has long been the most popular destination for shell company incorporation. This BFS asks: which EU countries appear most frequently as the origin of officers who control BVI-registered entities, and how does the size of that flow compare across countries?

The query starts from officers in five large EU economies (Portugal, Spain, France, Germany, Italy) and looks for paths of length 1 or 2 that end at any entity whose jurisdiction is BVI. A path of length 1 means the officer is directly listed on the entity. A path of length 2 means the connection passes through an intermediate node — typically a corporate vehicle or a name-deduplication link.

In [30]:
# Officers in EU countries (excluding tax havens) connected to BVI entities
bfs_eu_to_bvi = g.bfs(
    fromExpr="node_type = 'officer' AND countries IS NOT NULL AND "
             "(countries LIKE '%Portugal%' OR countries LIKE '%Spain%' OR "
             " countries LIKE '%France%' OR countries LIKE '%Germany%' OR "
             " countries LIKE '%Italy%')",
    toExpr="node_type = 'entity' AND jurisdiction = 'BVI'",
    maxPathLength=2
)

print(f"EU officers → BVI entities (1–2 hops): {bfs_eu_to_bvi.count()}")

print("\nOfficer country × intermediary chain → BVI entity:")
(bfs_eu_to_bvi
    .groupBy(F.col("from.countries").alias("officer_country"))
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=40))

EU officers → BVI entities (1–2 hops): 2469

Officer country × intermediary chain → BVI entity:


+----------------------------------------+-----+
|                         officer_country|count|
+----------------------------------------+-----+
|Italy;Canada;Australia;United States;...| 1570|
|                                  France|  196|
|Indonesia;Italy;British Virgin Island...|  173|
|                                   Spain|  137|
|                                   Italy|  124|
|                                Portugal|  111|
|                                 Germany|   88|
|                      Hong Kong;Portugal|   11|
|                       Germany;Hong Kong|   10|
|                        France;Hong Kong|    8|
|China;Hong Kong;New Zealand;United St...|    8|
|                         France;Thailand|    5|
|                           France;Mexico|    3|
|                          Austria;France|    3|
|                          Macao;Portugal|    2|
|                         Angola;Portugal|    2|
|                       Portugal;Thailand|    1|
|                   

The output gives us a country-by-country profile of EU exposure to BVI. Italy leads (124 single-country officers + 1,570 in mixed-nationality rows), followed by France (196), Spain (137), Portugal (111), and Germany (88). Portugal's footprint here is modest in absolute terms but is on the same order of magnitude as the larger EU economies, which is itself a finding — Portugal punches above its weight relative to its economic size.

The mixed-country rows (e.g., "Hong Kong;Portugal", "Macao;Portugal", "Angola;Portugal") are also worth flagging: they identify officers whose listed nationality spans multiple countries, frequently linking Portugal with its former colonies or with Hong Kong/Macao. These are exactly the multi-jurisdictional individuals that would be hard to surface by attribute filtering alone.

A reporter would now want to drill into specific officers in the rows above — particularly the Portugal-only and Portugal-mixed rows — to see who they are and what entities they connect to.

### BFS 2 — Address hubs serving many nationalities

Physical addresses in offshore jurisdictions are often shared by hundreds or thousands of entities — a single building in Bermuda might be the registered address for companies controlled by officers from dozens of countries. These shared addresses are operational hubs: somebody at that address is providing corporate services, and the breadth of nationalities passing through tells us about the scale and reach of the operation.

This BFS traces officer → entity → address paths and aggregates by destination address, counting how many distinct officer nationalities arrive at each one.

In [8]:
officer_to_addr = g.bfs(
    fromExpr="node_type = 'officer' AND countries IS NOT NULL",
    toExpr="node_type = 'address'",
    maxPathLength=2
).cache()

# Inspect what BFS actually produced
print("BFS columns:", officer_to_addr.columns)
officer_to_addr.printSchema()

26/05/17 20:28:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


BFS columns: ['from', 'e0', 'to']
root
 |-- from: struct (nullable = false)
 |    |-- id: string (nullable = true)
 |    |-- display_name: string (nullable = true)
 |    |-- node_type: string (nullable = false)
 |    |-- name: string (nullable = true)
 |    |-- address: string (nullable = true)
 |    |-- countries: string (nullable = true)
 |    |-- country_codes: string (nullable = true)
 |    |-- jurisdiction: string (nullable = true)
 |    |-- jurisdiction_description: string (nullable = true)
 |    |-- sourceID: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- service_provider: string (nullable = true)
 |    |-- valid_until: string (nullable = true)
 |-- e0: struct (nullable = false)
 |    |-- src: string (nullable = true)
 |    |-- dst: string (nullable = true)
 |    |-- rel_type: string (nullable = true)
 |    |-- rel_type_clean: string (nullable = true)
 |    |-- link: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- star

In [9]:
flat = officer_to_addr.select(
    F.col("from.id").alias("officer_id"),
    F.col("from.countries").alias("officer_country"),
    F.col("to.display_name").alias("address"),
    F.col("to.countries").alias("address_country"),
)

hub_addresses = (flat
    .groupBy("address", "address_country")
    .agg(
        F.countDistinct("officer_country").alias("distinct_officer_countries"),
        F.countDistinct("officer_id").alias("num_officers"),
    )
    .filter(F.col("distinct_officer_countries") >= 5)
    .orderBy(F.desc("distinct_officer_countries"))
)
hub_addresses.show(20, truncate=60)

26/05/17 20:29:43 WARN MemoryStore: Not enough space to cache rdd_243_2 in memory! (computed 68.0 MiB so far)
26/05/17 20:29:43 WARN MemoryStore: Not enough space to cache rdd_243_0 in memory! (computed 68.0 MiB so far)
26/05/17 20:29:43 WARN BlockManager: Persisting block rdd_243_2 to disk instead.
26/05/17 20:29:44 WARN BlockManager: Persisting block rdd_243_0 to disk instead.
26/05/17 20:29:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:29:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:29:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:29:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:29:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:29:46 WARN RowBasedKe

+------------------------------------------------------------+----------------------+--------------------------+------------+
|                                                     address|       address_country|distinct_officer_countries|num_officers|
+------------------------------------------------------------+----------------------+--------------------------+------------+
| Canon's Court; 22 Victoria Street; Hamilton; HM 12; Bermuda|               Bermuda|                        97|        1247|
|PO Box 887 GT; Clifton House; 75 Fort Street; George Town...|        Cayman Islands|                        79|         357|
|     Argyle House; 41a Cedar Avenue; Hamilton HM 12; Bermuda|               Bermuda|                        64|         879|
|Clifton House; 75 Fort Street; Grand Cayman KY1-1108; Cay...|        Cayman Islands|                        35|         720|
|                                  80, BROAD STREET' MONROVIA|                  NULL|                        32|      

The top of this list is dominated by addresses in Bermuda and the Cayman Islands, which is consistent with what we already know about offshore service providers. The most striking result is **Canon's Court, Hamilton, Bermuda**, which serves entities controlled by officers from 97 distinct nationality combinations across 1,247 officers. That is not a coincidence — it is the headquarters of a major offshore law firm operating at industrial scale.

What makes this insight specifically a BFS result rather than a simple join is the aggregation across the 2-hop path: we are not just counting "officers connected directly to this address", we are counting "officers whose entities are registered at this address, after passing through the intermediate entity node." Addresses don't connect to officers directly in this data; the relationship is mediated by entities, and BFS is what lets us aggregate across that hop cleanly.

For a reporter, this list is a directory of the most important nodes to investigate physically. Each row is a building, and at each building someone is running operations for clients from dozens of countries.

### BFS 3 — Portugal's two-hop neighbourhood

Portugal has only about 30 nodes directly tagged with the country label in the Panama Papers leak. That's a small footprint — but the more interesting question for a reporter is: *what else is connected to those 30 nodes within two hops?* In other words, what non-Portuguese parts of the offshore network sit in Portugal's immediate orbit?

This is the canonical use case for BFS: expand outward from a small seed set, stop after a fixed number of hops, and look at what you've collected. The query starts from any node whose `countries` field contains "Portugal" and walks up to two hops outward, retaining only the non-Portuguese nodes it reaches. We use the undirected version of the graph here so that we can traverse in either direction along the role-bipartite edges.

In [40]:
# Everything reachable from Portugal-linked nodes within 2 hops, excluding Portugal itself
portugal_orbit = g_undirected.bfs(
    fromExpr="countries IS NOT NULL AND countries LIKE '%Portugal%'",
    toExpr="(countries IS NULL OR countries NOT LIKE '%Portugal%')",
    maxPathLength=2
).cache()

print(f"Non-Portugal nodes within 2 hops of Portugal: {portugal_orbit.count():,}")

# What are these non-Portuguese nodes? Group by type and country
print("Non-Portuguese nodes in Portugal's 2-hop neighbourhood:")
(portugal_orbit
    .select(F.col("to.display_name").alias("name"),
            F.col("to.node_type").alias("type"),
            F.col("to.countries").alias("country"),
            F.col("to.jurisdiction_description").alias("jurisdiction"))
    .distinct()
    .groupBy("type", "country")
    .count()
    .orderBy(F.desc("count"))
    .show(30, truncate=40))

# Specific named nodes in the orbit — for a reporter to investigate manually
print("\nNamed intermediaries in Portugal's orbit:")
(portugal_orbit
    .filter(F.col("to.node_type") == "intermediary")
    .select(F.col("to.display_name").alias("intermediary"),
            F.col("to.countries").alias("country"))
    .distinct()
    .show(30, truncate=60))

Non-Portugal nodes within 2 hops of Portugal: 7,268
Non-Portuguese nodes in Portugal's 2-hop neighbourhood:
+-------+----------------------+-----+
|   type|               country|count|
+-------+----------------------+-----+
|address|                  NULL| 1149|
| entity|                 Malta| 1113|
| entity|                  NULL|  179|
| entity|British Virgin Islands|  165|
|officer|                  NULL|   53|
| entity|               Bermuda|   45|
|address|                   PRT|   41|
| entity|           Switzerland|   23|
| entity|           Netherlands|   22|
| entity|                Jersey|   19|
|officer|            Luxembourg|   14|
|officer|British Virgin Islands|   13|
|officer|               Bahamas|   12|
|address|             Hong Kong|   11|
|officer|           Switzerland|   11|
| entity|             Hong Kong|   10|
|officer|                Angola|   10|
| entity|        Cayman Islands|   10|
| entity|                 Qatar|   10|
| entity|        Not identified|  

Within two hops of Portugal's ~30 nodes, BFS surfaces over 7,000 non-Portuguese nodes. The composition is revealing:

- **Malta dominates the entity side** with 1,113 entities — far more than any other jurisdiction, and more than four times the next-largest country. This points to Malta as a primary offshore destination for Portuguese-linked structures, not BVI or Cayman as one might expect by default.
- **BVI is second** (165 entities), followed by Bermuda (45), Switzerland (23), Netherlands (22), and Jersey (19) — a familiar list of European offshore jurisdictions.
- **Officers in the orbit** come from Luxembourg, BVI, Bahamas, Switzerland, and notably **Angola** (10 officers) and other Portuguese-speaking jurisdictions — a pattern consistent with the lusophone connection visible in BFS 1.
- The **named intermediaries** are the most actionable finding for a reporter. The list includes both institutional service providers (Appleby Trust Cayman, Close Trustees Guernsey) and named individuals (Pedro Afra Rosa, Ana Rosa Ferreira Moutinho, Pedro Miguel Andrade Mendes). These named individuals are the human-scale starting points for follow-up reporting — they are non-Portuguese-tagged intermediaries whose work brings them within two hops of Portuguese activity.

The Malta finding is the surprising one. Standard journalistic intuition about Portuguese offshore activity would point toward Atlantic jurisdictions (BVI, Bermuda) or perhaps Switzerland; the dominance of Malta is not something the country-level statistics or the degree centrality results would have surfaced on their own. This is what BFS is for: making the structure of the local neighbourhood visible.

## Shortest paths

Shortest paths is one of the foundational distance measures in graph theory. Given a node, it asks: what is the minimum number of edges you'd have to traverse to reach every other node in the graph? The "minimum" is essential — it gives us a single, well-defined number for the distance between any two nodes, regardless of how many alternative paths exist between them.

### The general idea

If a graph were a road network and edges were roads, the shortest path from your home to a destination would tell you the fewest roads you need to drive on to get there. In an unweighted graph (where all edges are treated equally), this is computed using BFS from each landmark — but framed as a question about distance rather than reachability.

### When to use shortest paths

Shortest paths is the right tool when you want to:

- **Measure centrality through reach**: nodes close to many other nodes are structurally central
- **Identify peripheral or core members**: a node's average distance to a reference point tells you whether it sits in the dense middle of the network or out on the edges
- **Reveal geographic or organizational "shells"**: at distance 1 from a hub you find its direct neighbours, at distance 2 you find its neighbours' neighbours, and so on. Each shell can have a distinct character.
- **Compare populations on the same scale**: by computing distances from every node to a fixed landmark, you can place very different subsets of the graph (e.g., officers from different countries) on a common ruler.

### Shortest paths in GraphFrames

GraphFrames computes shortest paths from every node to one or more **landmark** nodes. The output is a column called `distances` containing a map of `{landmark_id → distance}` for each node. Nodes that cannot reach any landmark have an empty map.

There are two important details to internalize:

1. **GraphFrames respects edge direction.** A directed graph may give very different distances than its undirected version. In our case, the role-bipartite structure means we will use the undirected graph for distance computations.
2. **Reachability is not guaranteed.** If the graph is fragmented into multiple connected components, only nodes in the same component as the landmark will have a defined distance. Always restrict the analysis to the largest connected component to avoid mostly-empty results.

### Applying shortest paths to the Panama Papers

For our investigation, shortest paths lets us measure how "central" different parts of the network are relative to a chosen hub. We will use the highest-degree node in the largest connected component as our landmark — this is the node most likely to anchor a meaningful distance ruler. We can then ask: which officers and intermediaries sit close to this hub? What does the geographic composition look like at each distance? And specifically, where do Portugal-linked nodes fall on that ruler?|

### Step 1 — Restrict to the largest connected component, then pick a landmark

Before computing shortest paths, we need to ensure that the landmark is reachable from a useful fraction of the graph. The Panama Papers leak is not a single connected blob — it fragments into many disconnected pieces, with most nodes living in one giant component and the rest in smaller islands.

Our approach is to:
1. Compute connected components on the undirected version of the graph
2. Identify the largest one (the giant component)
3. Restrict our subsequent analysis to nodes inside that component
4. Pick the highest-degree node in that component as our landmark

The result is a subgraph where every node is guaranteed to have a defined distance to the landmark — eliminating the noise of nodes that simply cannot be reached.

In [33]:
# Step 1: Get the largest connected component (you likely already have CC from Week 1;
# if not, compute it here on g_undirected)
cc = g_undirected.connectedComponents().cache()

largest_cc_id = (cc.groupBy("component").count()
    .orderBy(F.desc("count")).first()["component"])

largest_cc_ids = cc.filter(F.col("component") == largest_cc_id).select("id")
print(f"Largest CC size: {largest_cc_ids.count():,} nodes")

# Step 2: Build the largest-CC subgraph (vertices + induced edges)
lcc_vertices = g.vertices.join(largest_cc_ids, on="id", how="inner")

lcc_edges = (g_undirected.edges
    .join(largest_cc_ids.withColumnRenamed("id", "src"), on="src", how="inner")
    .join(largest_cc_ids.withColumnRenamed("id", "dst"), on="dst", how="inner")
)
g_lcc = GraphFrame(lcc_vertices, lcc_edges)

# Step 3: Top-degree node in the LCC is our landmark
landmark_row = (g_lcc.degrees.orderBy(F.desc("degree")).limit(1)
    .join(g.vertices, on="id").collect()[0])

landmark_id = landmark_row["id"]
print(f"Landmark: {landmark_row['display_name']}  "
      f"(type={landmark_row['node_type']}, country={landmark_row['countries']})")

26/05/17 20:18:12 WARN ConnectedComponents$: Returned DataFrame is persistent and materialized!
26/05/17 20:18:12 WARN CacheManager: Asked to cache already cached data.


Largest CC size: 17,363 nodes


Landmark: Portcullis TrustNet (BVI) Limited  (type=officer, country=Thailand;British Virgin Islands;Indonesia;Singapore)


The landmark is **Portcullis TrustNet (BVI) Limited** — a Caribbean-based corporate services provider that has appeared repeatedly in our previous analyses (top of the PageRank, top intermediaries by out-degree). It sits at distance 0 to itself and acts as a natural anchor point for measuring how the rest of the network is organized around it.

### Step 2 — Compute shortest paths from every node to the landmark

Now we run the algorithm. The output is a `distances` map per node; we collapse it to a single integer column `dist` (the distance to our single landmark), then look at how distances are distributed across the graph.

In [34]:
# Now run shortest paths INSIDE the largest CC — guaranteed reachability
sp = g_lcc.shortestPaths(landmarks=[landmark_id]).cache()

sp = sp.withColumn("dist", F.array_min(F.map_values("distances")))

# Sanity check
print("Distance distribution to the landmark:")
sp.filter(F.col("dist").isNotNull()).groupBy("dist").count() \
   .orderBy("dist").show(20)

26/05/17 20:18:43 WARN ShortestPaths: Returned DataFrame is persistent and materialized!
26/05/17 20:18:43 WARN CacheManager: Asked to cache already cached data.


Distance distribution to the landmark:
+----+-----+
|dist|count|
+----+-----+
|   0|    1|
|   1| 7400|
|   2| 3154|
|   3| 1608|
|   4|  598|
|   5|  277|
|   6|  145|
|   7| 1663|
|   8|  185|
|   9|   32|
|  10|   45|
|  11|   10|
|  12|  313|
|  13|  192|
|  14|   36|
|  15|   39|
|  16|  130|
|  17|   52|
|  18|  555|
|  19|  128|
+----+-----+
only showing top 20 rows



The distance distribution is informative in its own right. The bulk of reachable nodes sit within 1–4 hops of the landmark — these are the entities, addresses, and officers in the immediate operational orbit of Portcullis TrustNet. The "spike" at distance 7 (1,663 nodes) is interesting: it suggests a second cluster of nodes that connects back to the landmark through a long chain, possibly via a different intermediary serving as a bridge. The long tail extending past distance 18 indicates that some parts of the giant component are surprisingly far from the main hub — these are the peripheral edges of the offshore network, connected only through long indirect chains.

### Investigative angle 1 — Who sits closest to the central hub?

For an investigative reporter, the most useful question is: which *people* (officers and intermediaries) sit closest to the network's most-connected node? These are the structural insiders — individuals whose offshore activity is tightly intertwined with the main hub of the leak.

In [35]:
# Investigative angle 1: "Closeness" — which nodes are unexpectedly close to the hub?
# Officers/intermediaries with small distance are tightly embedded brokers
print("Officers and intermediaries closest to the network's most-connected node:")
(sp.filter(F.col("dist").isNotNull() &
           F.col("node_type").isin("officer", "intermediary"))
   .select("display_name", "node_type", "countries", "dist")
   .orderBy("dist", F.desc("node_type"))
   .show(30, truncate=60))

Officers and intermediaries closest to the network's most-connected node:
+---------------------------------+---------+---------------------------------------------------+----+
|                     display_name|node_type|                                          countries|dist|
+---------------------------------+---------+---------------------------------------------------+----+
|Portcullis TrustNet (BVI) Limited|  officer|Thailand;British Virgin Islands;Indonesia;Singapore|   0|
|                   Chiang, Hsi Wu|  officer|                                     Not identified|   2|
|       Profit Index (H.K) Limited|  officer|                                     Not identified|   2|
|                   Helen Fon Chen|  officer|                                     Not identified|   2|
|                     LEE TAK MING|  officer|                                     Not identified|   2|
|                     Mermeden Ltd|  officer|                      Indonesia;Hong Kong;Australia|   2|

The list at distance 2 is dominated by officers tagged "Not identified" — a reflection of how the Panama Papers data records officers whose nationality the ICIJ could not determine. Among those with country information, the closest officers come from Hong Kong, Indonesia, Singapore, South Korea, and Taiwan, with a few outliers from Venezuela and other regions. This is consistent with Portcullis TrustNet's known client base, which historically focused heavily on Southeast Asian clientele.

The notable presence of "To Be Assigned" as an officer is worth flagging — this kind of placeholder name suggests entities created in advance with directors yet to be filled in, a hallmark of shell company production at scale.

### Investigative angle 2 — Geographic "shells" around the hub

A more macro-level question: as we move outward from the hub, how does the country composition change? At distance 1 we'd expect the immediate operational geography (BVI itself); at greater distances, we expect to see the geographic origins of the people whose entities are managed by the hub.

This view exposes the *layered structure* of the offshore network: jurisdictions of incorporation at the inner shells, beneficial-owner geographies at the outer shells.

In [36]:
# Investigative angle 2: Geographic "shells" around the hub
# At distance 1 you'd expect Panama/BVI; the question is what shows up at distance 3, 4, 5
print("Country composition by distance from the hub (top per distance):")
country_shells = (sp
    .filter(F.col("dist").isNotNull() &
            (F.col("dist") <= 5) &
            F.col("countries").isNotNull())
    .groupBy("dist", "countries")
    .count()
    .filter(F.col("count") >= 20)
)

from pyspark.sql.window import Window
w = Window.partitionBy("dist").orderBy(F.desc("count"))
country_shells.withColumn("rank", F.row_number().over(w)) \
    .filter(F.col("rank") <= 5) \
    .orderBy("dist", "rank") \
    .show(40, truncate=False)

Country composition by distance from the hub (top per distance):
+----+-------------------------------------+-----+----+
|dist|countries                            |count|rank|
+----+-------------------------------------+-----+----+
|1   |British Virgin Islands               |7110 |1   |
|1   |Cayman Islands;British Virgin Islands|82   |2   |
|1   |British Virgin Islands;Cayman Islands|75   |3   |
|1   |British Virgin Islands;Singapore     |33   |4   |
|2   |Not identified                       |814  |1   |
|2   |Hong Kong                            |724  |2   |
|2   |Taiwan                               |383  |3   |
|2   |China                                |317  |4   |
|2   |Singapore                            |216  |5   |
|3   |Not identified                       |421  |1   |
|3   |Samoa                                |227  |2   |
|3   |Hong Kong                            |155  |3   |
|3   |Singapore                            |85   |4   |
|3   |Samoa;Cayman Islands             

The shells reveal a clear pattern:

- **Distance 1** is overwhelmingly BVI (7,110 nodes) — these are the entities directly registered through Portcullis TrustNet. This is the layer of *incorporation*.
- **Distance 2** shifts to Hong Kong, Taiwan, China, and Singapore — the **client geography**. These are the addresses and officers connected to those BVI entities, revealing where the actual beneficial owners and operators are based.
- **Distances 3–5** extend further into Samoa, Malaysia, the United States, and Cayman Islands — secondary client regions and, at distance 5, an unexpected concentration in Samoa (likely entities incorporated in Samoa as alternatives to BVI).

The progression from "jurisdiction" → "client country" → "secondary networks" is a textbook offshore structure made visible by distance. A reporter looking at this would conclude that Portcullis TrustNet's operations are primarily an East/Southeast Asian phenomenon, with a long tail extending into other jurisdictions.

### Investigative angle 3 — Where does Portugal fall on this ruler?

Finally, the targeted question: where do Portugal-linked nodes sit relative to this hub? Are they in the inner shells (operationally close to a major intermediary) or far out on the periphery?

In [37]:
# Investigative angle 3: Portugal's position
# How far are Portugal-linked nodes from the network's central hub?
# (Answers: are they central or peripheral?)
print("Portugal-linked nodes and their distance to the network hub:")
(sp.filter(F.col("dist").isNotNull() &
           F.col("countries").isNotNull() &
           F.col("countries").contains("Portugal"))
   .select("display_name", "node_type", "countries",
           "jurisdiction_description", "dist")
   .orderBy("dist")
   .show(50, truncate=60))

Portugal-linked nodes and their distance to the network hub:
+-------------------------+---------+---------+------------------------+----+
|             display_name|node_type|countries|jurisdiction_description|dist|
+-------------------------+---------+---------+------------------------+----+
|Mrs Gladys Toshiko Shanks|  officer| Portugal|                    NULL|   9|
|            Farida Ismail|  officer| Portugal|                    NULL|  19|
+-------------------------+---------+---------+------------------------+----+



Only two Portugal-tagged officers appear in this component, at distances 9 and 19 from the hub. Both are *far* from Portcullis TrustNet — distance 9 places this officer well beyond the immediate operational orbit, and distance 19 is essentially on the opposite edge of the giant component.

This is itself a finding: **Portuguese-tagged officers are not part of Portcullis TrustNet's operational network**. Combined with our earlier BFS results (where Portugal's strongest connections were to Malta, BVI through other channels, and lusophone Africa), this suggests that Portuguese offshore activity in the leak passes through different intermediaries entirely — not the largest hub of the Panama Papers, but a separate, smaller cluster of providers serving lusophone clients.

A reporter following this thread would now want to identify *which* intermediary anchors that Portuguese cluster, and use shortest paths with that node as a landmark instead.

## Label propagation — community detection

Label propagation (LPA) is one of the simplest community detection algorithms in graph theory. It works by an intuitive principle: every node looks at its neighbours and adopts the label that is most common among them. Repeat this enough times and stable clusters emerge — groups of nodes that all share the same label because they are densely connected to each other and only sparsely connected to the rest of the graph.

### The general idea

If you imagine a graph as a social network and edges as friendships, label propagation is like watching opinions spread through clusters. People adopt the dominant opinion of their friend group, and over time the network settles into distinct opinion communities. These communities are not defined by any explicit attribute — they emerge purely from the structure of who is connected to whom.

### When to use community detection

Label propagation is the right tool when you want to:

- **Discover natural groupings** in the graph without specifying what defines a group
- **Identify operational clusters** — groups of nodes that work together more than they work with the rest of the network
- **Compare structural communities with metadata communities** — does the algorithm group nodes by country? By jurisdiction? By something else entirely?
- **Find unexpected co-locations** — nodes that share a community despite having no obvious attribute in common

### Caveats

Label propagation is fast and scalable but has a known limitation: **it is stochastic**. Running it twice can give different community assignments because of ties in the "most common neighbour label" step. It also tends to produce a few very large communities and many tiny ones — including many singleton communities for nodes with no neighbours. Treat the results as a structural hint, not a definitive partition.

The labels themselves are arbitrary integers. Their value lies entirely in *what nodes share a label*, not in the label number itself.

### Applying LPA to the Panama Papers

For our investigation, LPA lets us discover clusters of nodes that operate together — possibly under shared intermediaries, possibly serving clients from the same country, possibly using the same jurisdiction. Once we have the communities, the real work is profiling them: what kind of nodes inhabit each one, and what does each community represent operationally?

### Step 1 — Run LPA and look at the community size distribution

We run label propagation with 5 iterations, which is typically enough for convergence on large graphs. The output assigns every node a `label` — an integer identifying its community.

In [10]:
lpa_results = g.labelPropagation(maxIter=5).cache()

community_sizes = (
    lpa_results
    .groupBy("label")
    .agg(F.count("*").alias("community_size"))
    .orderBy(F.desc("community_size"))
)

community_sizes.show(20, truncate=False)


26/05/17 20:31:36 WARN LabelPropagation: Returned DataFrame is persistent and materialized!
26/05/17 20:31:36 WARN CacheManager: Asked to cache already cached data.


+-----------+--------------+
|label      |community_size|
+-----------+--------------+
|17180214552|4497          |
|8590225402 |1562          |
|8590275181 |1350          |
|8590391134 |1257          |
|17180158913|1013          |
|25770155929|730           |
|8590225412 |698           |
|8590100988 |584           |
|80592      |582           |
|272937     |544           |
|25770111466|541           |
|407170     |493           |
|17179966996|489           |
|466436     |485           |
|8590225398 |457           |
|405964     |456           |
|25770158903|447           |
|272970     |327           |
|25770111446|302           |
|25770113599|296           |
+-----------+--------------+
only showing top 20 rows



The size distribution shows the typical LPA pattern: a few large communities (the largest with 4,497 nodes) and many smaller ones tailing off rapidly. The community labels are arbitrary numbers (some very large, some small) — they have no inherent meaning. The interesting work begins when we profile what's inside each community.

In [11]:
# After Spark restart and re-running LPA:
lpa_vertices = (lpa_results
    .select("id", "label", "node_type", "countries",
            "jurisdiction_description", "display_name", "sourceID")
    .cache())

# Now all the downstream profile / concentration / Portugal / intermediary
# analyses work as written.

### Step 2 — Profile the top 10 communities by node type

For each of the largest 10 communities, what is the breakdown of node types? A community that is mostly entities is structurally different from one that is mostly officers. By pivoting the data on `node_type`, we get a quick "fingerprint" of each large community.

In [12]:
# Profile the top 10 communities
top_labels = [r["label"] for r in community_sizes.limit(10).collect()]

community_profile = (lpa_vertices
    .filter(F.col("label").isin(top_labels))
    .groupBy("label", "node_type")
    .count()
    .groupBy("label")
    .pivot("node_type")
    .sum("count")
    .join(community_sizes, on="label")
    .orderBy(F.desc("community_size"))
)
community_profile.show(truncate=False)

+-----------+-------+------+------------+-------+--------------+
|label      |address|entity|intermediary|officer|community_size|
+-----------+-------+------+------------+-------+--------------+
|17180214552|37     |4457  |NULL        |3      |4497          |
|8590225402 |1      |1558  |NULL        |3      |1562          |
|8590275181 |NULL   |1350  |NULL        |NULL   |1350          |
|8590391134 |28     |1077  |NULL        |152    |1257          |
|17180158913|NULL   |1013  |NULL        |NULL   |1013          |
|25770155929|NULL   |730   |NULL        |NULL   |730           |
|8590225412 |2      |696   |NULL        |NULL   |698           |
|8590100988 |5      |3     |125         |451    |584           |
|80592      |NULL   |582   |NULL        |NULL   |582           |
|272937     |1      |543   |NULL        |NULL   |544           |
+-----------+-------+------+------------+-------+--------------+



The profiles reveal three distinct structural patterns:

- **Entity-dominated communities** (17180214552, 8590225402, 8590275181, 17180158913, 25770155929, 80592, 272937): communities where almost all the nodes are entities, with very few officers or addresses. These represent clusters of incorporations that share little connective tissue — many shell companies registered by the same provider but with mostly distinct officers/addresses.
- **Officer-heavy community** (8590100988): the only large community where officers dominate (451 officers, 125 intermediaries, only 3 entities). This is structurally different — a network of *people* tied together by overlapping intermediaries, rather than a cluster of entities.
- **Mixed communities** (8590391134): entities + officers + addresses in meaningful numbers, representing a more complete operational ecosystem.

The contrast between "entity-only" and "officer-heavy" communities is the most striking finding. The bulk of the offshore world is incorporated through providers who set up many entities for many different (largely disconnected) clients — hence the entity-only clusters. The officer-heavy community is the exception that proves the rule: it represents a network where the *people* are tightly interconnected, which is a much rarer and more investigatively interesting pattern.

### Step 3 — Country concentration: are communities national or international?

For each community, what is the dominant country, and what fraction of the community shares that country tag? A community where the top country accounts for 95% of nodes is a *national cluster*. One where the top country accounts for only 5% is an *international cluster*. Both are interesting, but they tell different stories.

In [13]:
# Top country per community + how concentrated
country_per_community = (lpa_vertices
    .filter(F.col("label").isin(top_labels) & F.col("countries").isNotNull())
    .groupBy("label", "countries")
    .count()
)

w = Window.partitionBy("label").orderBy(F.desc("count"))
top_country = (country_per_community
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") == 1)
    .select("label",
            F.col("countries").alias("top_country"),
            F.col("count").alias("top_country_count"))
)

community_concentration = (community_sizes
    .filter(F.col("label").isin(top_labels))
    .join(top_country, on="label")
    .withColumn("top_country_share",
                F.round(F.col("top_country_count") / F.col("community_size"), 3))
    .orderBy(F.desc("community_size"))
)
community_concentration.show(truncate=False)

+-----------+--------------+--------------------------------+-----------------+-----------------+
|label      |community_size|top_country                     |top_country_count|top_country_share|
+-----------+--------------+--------------------------------+-----------------+-----------------+
|17180214552|4497          |British Virgin Islands          |4288             |0.954            |
|8590225402 |1562          |Portugal                        |1                |0.001            |
|8590275181 |1350          |Bahamas                         |1350             |1.0              |
|8590391134 |1257          |Cayman Islands                  |1003             |0.798            |
|25770155929|730           |British Virgin Islands;Hong Kong|223              |0.305            |
|8590225412 |698           |Spain                           |1                |0.001            |
|8590100988 |584           |Not identified                  |193              |0.33             |
|80592      |582    

The concentration scores are illuminating:

- **Highly concentrated national communities**: community 8590275181 is 100% Bahamas, community 80592 is 98.5% Switzerland, community 17180214552 is 95.4% BVI. These are clusters where the offshore activity is geographically anchored — almost certainly the work of a single regional service provider serving a single national client base.
- **International / diffuse communities**: communities 8590225402 (0.1% top-country share) and 8590225412 (0.1% top-country share) are essentially borderless — the top country (Portugal in one case, Spain in the other) is a tiny fraction. Looking at these scores, the algorithm has detected a structural community that *exists* but whose members are spread across many countries. The "top country" label here is almost meaningless; the community is held together by structure, not nationality.
- **Mixed dual-country communities**: community 25770155929 is 30.5% "British Virgin Islands;Hong Kong" — a community that connects the BVI jurisdiction with Hong Kong-based clients in roughly equal measure. This is the classic offshore pipeline made visible as a single community.

The community-level concentration is a useful triage tool: an investigator who wants to study a *single* national offshore footprint would focus on the highly-concentrated communities, while one interested in cross-border operations would investigate the diffuse ones.

### Step 4 — Which communities contain Portugal-linked nodes?

Now the targeted question. Portugal has only about 30 nodes in the leak. Where does LPA place them? Are they scattered across many communities, or concentrated in a few specific ones?

In [14]:
# What community/communities do Portugal-linked nodes belong to?
portugal_communities = (lpa_vertices
    .filter(F.col("countries").isNotNull() &
            F.col("countries").contains("Portugal"))
    .groupBy("label")
    .agg(F.count("*").alias("portugal_nodes_in_community"))
    .join(community_sizes, on="label")
    .withColumn("portugal_share",
                F.round(F.col("portugal_nodes_in_community") / F.col("community_size"), 4))
    .orderBy(F.desc("portugal_nodes_in_community"))
)
portugal_communities.show(truncate=False)

+-----------+---------------------------+--------------+--------------+
|label      |portugal_nodes_in_community|community_size|portugal_share|
+-----------+---------------------------+--------------+--------------+
|292416     |8                          |8             |1.0           |
|135909     |8                          |12            |0.6667        |
|135648     |7                          |7             |1.0           |
|135910     |6                          |6             |1.0           |
|329737     |4                          |9             |0.4444        |
|335756     |2                          |20            |0.1           |
|113101     |2                          |2             |1.0           |
|442937     |2                          |3             |0.6667        |
|292191     |2                          |2             |1.0           |
|17180167527|2                          |2             |1.0           |
|35109      |2                          |2             |1.0     

Portugal-linked nodes are scattered across many small communities, with the largest concentration being just 8 nodes (community 292416). Most communities containing Portuguese nodes are tiny (size 1–12), and a striking pattern emerges: many of these communities are *exclusively* Portuguese (share = 1.0).

This is a meaningful structural finding. Rather than being embedded inside larger international offshore networks, Portuguese-linked nodes form their own small, isolated micro-clusters. There is no "Portugal community" of any significant size — instead there are many micro-communities, each containing only a handful of nodes.

Two interpretations are consistent with this:
1. Portuguese offshore activity in the leak passes through small, specialized intermediaries who handle a handful of Portuguese clients each, rather than through a large national-scale operation.
2. The Portuguese footprint in the leak is genuinely small and fragmented at the level of structural communities, even if individual cases may be significant.

A reporter would want to look at communities 292416, 135909, and 135648 in detail — the Portugal-dominated ones with multiple members — to understand what specific operations they represent.

### Step 5 — Which intermediaries serve many communities?

If an intermediary appears in the data multiple times under slightly different name variants (a common artifact in this kind of leak), LPA may place each variant in a different community. But beyond that, the question is real: are some intermediaries truly cross-cutting — serving clients across many distinct community clusters — while others serve only one?

In [16]:
intermediary_reach = (lpa_vertices
    .filter(F.col("node_type") == "intermediary")
    .groupBy("display_name")
    .agg(F.countDistinct("label").alias("num_communities"),
         F.count("*").alias("intermediary_node_count"))
    .filter(F.col("intermediary_node_count") >= 2)
    .orderBy(F.desc("num_communities"))
)
intermediary_reach.show(20, truncate=60)

+----------------------------------------+---------------+-----------------------+
|                            display_name|num_communities|intermediary_node_count|
+----------------------------------------+---------------+-----------------------+
|                       SMITH JENNIFER G.|             11|                     11|
|                     HUTCHINSON GAYLE A.|             10|                     10|
|                 HARRIDYAL-SODHA LIZA A.|             10|                     10|
|                   BOURQUE MARY ELLEN M.|              8|                      8|
|                        SMITH JENNIFER G|              5|                      5|
|                    CARMICHAEL TREVOR A.|              4|                      4|
|                ARTHUR-SELMAN JAMAR P.R.|              4|                      4|
|                     HUTCHINSON IAN STC.|              4|                      4|
|          CARMICHAEL, Q.C. DR. TREVOR A.|              4|                      4|
|   

The top of this list is dominated by intermediaries from Barbados-area legal firms (SMITH JENNIFER G., HUTCHINSON GAYLE A., HARRIDYAL-SODHA LIZA A., BOURQUE MARY ELLEN M.) — individual lawyers each appearing across 8–11 different communities. These are *generalist* service providers who interact with many distinct operational clusters in the leak.

The fact that the same intermediary appears across multiple communities means LPA found that her clients fall into structurally distinct groups — different sets of entities, with different officers and different addresses. The intermediary is the only thing tying them together, which is exactly the pattern we'd expect from a service-provider relationship: many independent clients, all routed through one professional.

The variation in name spelling for the same person (SMITH JENNIFER G. vs SMITH JENNIFER G, CARMICHAEL TREVOR A. vs CARMICHAEL TREVOR A) is also a useful reminder of the data quality issues in real-world leaks. A more careful analysis would deduplicate these variants before counting.

### Step 6 — Top jurisdiction per community

Finally, for each large community, where are the entities incorporated? This adds another layer to the community profile: country tags us where *people* are based, while jurisdiction tells us where *entities* are filed.

In [17]:
jurisdiction_per_community = (lpa_vertices
    .filter(F.col("label").isin(top_labels) &
            (F.col("node_type") == "entity") &
            F.col("jurisdiction_description").isNotNull())
    .groupBy("label", "jurisdiction_description")
    .count()
)

w = Window.partitionBy("label").orderBy(F.desc("count"))
(jurisdiction_per_community
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= 3)
    .orderBy("label", "rank")
    .show(50, truncate=False))

+-----------+------------------------+-----+----+
|label      |jurisdiction_description|count|rank|
+-----------+------------------------+-----+----+
|80592      |British Virgin Islands  |377  |1   |
|80592      |Panama                  |129  |2   |
|80592      |Bahamas                 |72   |3   |
|272937     |Bahamas                 |543  |1   |
|8590100988 |British Virgin Islands  |3    |1   |
|8590225402 |Bahamas                 |1558 |1   |
|8590225412 |Bahamas                 |696  |1   |
|8590275181 |Bahamas                 |1350 |1   |
|8590391134 |Cayman Islands          |987  |1   |
|8590391134 |British Virgin Islands  |26   |2   |
|8590391134 |State of Delaware       |18   |3   |
|17180158913|Bahamas                 |1013 |1   |
|17180214552|British Virgin Islands  |4410 |1   |
|17180214552|Undetermined            |16   |2   |
|17180214552|Samoa                   |16   |3   |
|25770155929|Undetermined            |730  |1   |
+-----------+------------------------+-----+----+


The jurisdiction profiles reinforce the community fingerprints we built earlier:

- **Single-jurisdiction communities** (8590225402, 8590225412, 8590275181, 17180158913 → all Bahamas; 17180214552 → BVI; 25770155929 → "Undetermined"): each community is a stream of incorporations through a single jurisdiction. The provider behind each of these is likely a single firm operating at scale in one place.
- **Multi-jurisdiction communities** (80592, 8590391134): communities that span multiple incorporation jurisdictions. Community 80592 splits across BVI (377), Panama (129), and Bahamas (72) — a sign that the provider behind this community operates across multiple offshore jurisdictions rather than specializing in one.

For a reporter, this view is the bridge between LPA's structural communities and the macro narrative of "which jurisdictions are most active in which kinds of operations." A community that is 100% Bahamas with 1,500+ entities is a single industrial operation in the Bahamas; a community that splits across three jurisdictions is something more sophisticated.

## Triangle count — finding tight clusters and rings

A triangle in a graph is three nodes that are all mutually connected: every node is a neighbour of every other node in the triple. Triangle count is exactly what it sounds like — for each node, how many triangles include it?

### The general idea

In a social network, a triangle means three people who all know each other. In a citation network, a triangle means three papers that all cite each other. In a road network, a triangle is three intersections connected by direct roads. The presence of triangles is a structural sign of **tight clustering** — three nodes that are not just pairwise connected but form a closed group.

The fraction of triangles a node participates in (relative to the number of triangles possible) is related to the **clustering coefficient**, a classical measure of how cliquey a node's neighbourhood is.

### When to use triangle count

Triangle count is the right tool when you want to:

- **Detect cliques or rings** — groups of nodes that all reinforce each other's connections
- **Measure local clustering** — distinguish nodes embedded in tight groups from nodes that act as bridges between otherwise separate parts of the graph
- **Find collusion or coordination signals** — in financial or social networks, dense triangles often indicate organized rather than incidental relationships

### The bipartite caveat

There is a subtle but critical point: **triangle count gives near-zero results on bipartite graphs**. A bipartite graph is one where nodes split into two groups and edges only go between groups, never within. Our Panama Papers graph is essentially bipartite by role — officers and intermediaries connect to entities and addresses, but never directly to each other. So three officers cannot form a triangle through the original edges, because officers are not directly connected.

The investigative version of triangle count on bipartite data is to first **project** the graph onto a single node type, then count triangles on the projection. The projection collapses paths through the other node type into direct edges, and triangles in the projection acquire a meaningful interpretation.

### Applying triangle count to the Panama Papers

For our investigation, we project the bipartite officer-of relationships into an **officer-to-officer co-membership graph**: two officers are connected if they share at least one entity (a board they both sit on, or a company they both control). Triangles in this projection mean three officers who all overlap on boards with each other — a **board ring**.

To filter out noise, we require officer pairs to share at least *two* entities before we connect them. Two officers sharing a single board could be coincidence (they hire the same accountant); two officers sharing multiple boards represent a working relationship.

### Step 1 — Project to an officer-to-officer co-membership graph

We start by self-joining the `officer_of` edges on their target entity. Each match represents a pair of officers who share that entity. We then aggregate to count how many entities each pair shares, and keep only pairs sharing at least two entities.

In [18]:
# Step 1: Project the bipartite officer-of graph to an officer-officer co-membership graph
# Two officers are connected if they share at least one entity

officer_edges_raw = g.edges.filter(F.col("rel_type") == "officer_of")

# Self-join via the shared entity (dst)
o2o = (officer_edges_raw.alias("a")
    .join(officer_edges_raw.alias("b"),
          (F.col("a.dst") == F.col("b.dst")) & (F.col("a.src") < F.col("b.src")))
    .select(F.col("a.src").alias("officer_a"),
            F.col("b.src").alias("officer_b"),
            F.col("a.dst").alias("shared_entity"))
)

# Aggregate: how many entities does each officer pair share?
officer_pairs = (o2o
    .groupBy("officer_a", "officer_b")
    .agg(F.countDistinct("shared_entity").alias("shared_entities"))
    .filter(F.col("shared_entities") >= 2)   # noise threshold
    .cache()
)

print(f"Officer pairs sharing ≥2 entities: {officer_pairs.count():,}")
officer_pairs.show(10)

26/05/17 20:34:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:34:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:34:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:34:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:34:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:34:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:34:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:34:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/17 20:34:55 WARN RowBasedKeyValueBatch: Calling spill() on

Officer pairs sharing ≥2 entities: 322,609
+---------+---------+---------------+
|officer_a|officer_b|shared_entities|
+---------+---------+---------------+
|   100013|   100125|              2|
|   100013|   105404|              2|
|   100013|   106614|              2|
|   100013|    63407|              2|
|   100060|    67541|              2|
|   100075|   104293|              2|
|   100081|   118529|              2|
|   100081|    48072|              2|
|   100081|    54662|              2|
|   100091|   120542|              2|
+---------+---------+---------------+
only showing top 10 rows



The projection produces a substantial number of officer pairs sharing multiple entities. Each row in the output represents a pair of officers who have at least two boards in common — a non-trivial level of co-membership that suggests a real working relationship.

### Step 2 — Build the projected GraphFrame and count triangles

We now construct a new GraphFrame using only these officer pairs as edges. We add reverse edges to make the graph undirected (triangle count requires this), then run the algorithm with disk-fallback caching to handle the memory pressure that triangle count can create on large projections.

In [19]:
# Step 2: Build the projected GraphFrame
proj_vertex_ids = (officer_pairs.select(F.col("officer_a").alias("id"))
    .union(officer_pairs.select(F.col("officer_b").alias("id")))
    .distinct())

proj_vertices = g.vertices.join(proj_vertex_ids, on="id", how="inner")

# Make undirected by adding reverse edges
proj_edges_fwd = officer_pairs.select(
    F.col("officer_a").alias("src"),
    F.col("officer_b").alias("dst"),
    F.col("shared_entities"))
proj_edges_rev = officer_pairs.select(
    F.col("officer_b").alias("src"),
    F.col("officer_a").alias("dst"),
    F.col("shared_entities"))
proj_edges = proj_edges_fwd.unionByName(proj_edges_rev)

g_proj = GraphFrame(proj_vertices, proj_edges)
print(f"Officer co-membership graph: {g_proj.vertices.count():,} vertices, "
      f"{g_proj.edges.count():,} edges")

Officer co-membership graph: 77,521 vertices, 645,218 edges


The projected graph has 77,521 officers and 645,218 co-membership edges — a substantial subnetwork that captures the "people who work together repeatedly" structure of the Panama Papers.

The top results are striking. Michael J Burns (Bermuda/BVI/Canada) leads with 954 triangles, followed by KPMG Bermuda (842), Timothy J Counsell (807), and Warren Wilton Cabral (682). These names are not random — they appear repeatedly in connection with Bermuda corporate services, and the high triangle counts confirm structurally what those repeated appearances suggest: these are the people at the centre of densely interconnected rings of co-directorship.

**A critical pattern to notice:** below the top dozen entries, there is a long block of 22 officers all tied at exactly **325 triangles** — almost all listed in Jersey. This identical count is not a coincidence. It almost certainly reflects a single tight ring of co-directors who all sit on the same set of about 26 entities together. Every triple within this ring forms a triangle, and because the ring is so interconnected, every member appears in the same number of triangles (mathematically: $\binom{25}{2} = 300$, plus combinations from larger sets).

This is an important teaching point: **clusters of officers with identical triangle counts are a signature of a single ring, not independent findings**. The investigative story here is "one Jersey-based ring of about 26 people running 26 connected entities", not "26 separate notable officers". A reporter who saw this would treat the entire block as a single story, not 22 stories.

In [20]:
# Step 3: Triangle count on the projection
from pyspark import StorageLevel

tri = g_proj.triangleCount(storage_level=StorageLevel.MEMORY_AND_DISK)

print("Top officers by number of triangles (= number of 'rings' they sit in):")
(tri.filter(F.col("count") > 0)
   .select("id", "display_name", "countries", "count")
   .orderBy(F.desc("count"))
   .show(30, truncate=60))

26/05/17 20:35:33 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/05/17 20:35:45 WARN TriangleCount$: Returned DataFrame is persistent and materialized!


Top officers by number of triangles (= number of 'rings' they sit in):
+--------+------------------------------------------------------------+---------------------------------------------------+-----+
|      id|                                                display_name|                                          countries|count|
+--------+------------------------------------------------------------+---------------------------------------------------+-----+
|80042906|                                           Burns - Michael J|              Bermuda;British Virgin Islands;Canada|  954|
|80088213|                                              KPMG - Bermuda|                                            Bermuda|  842|
|80052824|                                        Counsell - Timothy J|               Bermuda;United Kingdom;United States|  807|
|80043562|                                      Cabral - Warren Wilton|                 Bermuda;Switzerland;United Kingdom|  682|
|   54662|         

### Investigative angle 1 — Are Portuguese officers in rings?

Now the targeted question. Are any of the Portugal-linked officers part of co-membership rings? The answer here is much more interesting if it's *low* than if it's high — a high triangle count for a Portuguese officer would mean they're embedded in a tight ring; a low count means they operate alone or in small pairs.

In [23]:
print("Portugal-linked officers and their ring membership:")
(tri.filter((F.col("count") > 0) &
            F.col("countries").isNotNull() &
            F.col("countries").contains("Portugal"))
   .select("display_name", "countries", "count")
   .orderBy(F.desc("count"))
   .show(truncate=60))

Portugal-linked officers and their ring membership:


+------------------------------+---------------------+-----+
|                  display_name|            countries|count|
+------------------------------+---------------------+-----+
|             de Frias - Duarte|     Bermuda;Portugal|    6|
|Pereira Viegas - Joao Henrique|             Portugal|    1|
|    JOAO MANUEL DA SILVA DE SA|South Africa;Portugal|    1|
+------------------------------+---------------------+-----+



Only three Portuguese-tagged officers appear in any triangle, and their counts are very low (1, 1, and 6 triangles). Compared to the top of the global list (954 triangles), Portuguese officers are at the *very bottom* of the ring structure.

This is consistent with our other findings: Portugal's footprint in the Panama Papers is structurally small and fragmented. Portuguese offshore activity does not appear to be organized as tight rings of co-directors; instead, Portuguese officers tend to appear individually or in pairs, without the dense interconnection that characterizes the Bermuda/BVI/Jersey rings. This could mean two things, both worth investigating:

1. Portuguese offshore activity is genuinely less organized into rings — more individual cases, less group structure.
2. The visible Portuguese activity in the leak is the tip of an iceberg whose ring structures sit elsewhere (e.g., under intermediaries that the leak captures less completely).

Duarte de Frias (Bermuda/Portugal, 6 triangles) is the closest thing to a Portuguese node in a ring, and his Bermuda connection suggests he is part of a Bermuda ring rather than a Portuguese one.

### Investigative angle 2 — Which countries have the most "ring-like" offshore activity?

Aggregating triangle counts by country gives us a global picture: which national footprints in the Panama Papers are organized into rings, and which are dominated by individual operators? A country with high average triangles per officer is one where offshore activity is *coordinated*; a country with low averages is one where it is *atomized*.

In [24]:
print("Average triangle count by officer country:")
(tri.filter((F.col("count") > 0) & F.col("countries").isNotNull())
   .groupBy("countries")
   .agg(F.count("*").alias("officers_in_rings"),
        F.round(F.avg("count"), 1).alias("avg_triangles"),
        F.max("count").alias("max_triangles"))
   .filter(F.col("officers_in_rings") >= 5)
   .orderBy(F.desc("avg_triangles"))
   .show(20, truncate=40))

Average triangle count by officer country:
+------------------------------------+-----------------+-------------+-------------+
|                           countries|officers_in_rings|avg_triangles|max_triangles|
+------------------------------------+-----------------+-------------+-------------+
|Bermuda;United Kingdom;United States|               16|        131.0|          807|
|                              Jersey|               67|         54.9|          429|
|                      Bermuda;Canada|               16|         46.9|          258|
|              Bermuda;United Kingdom|               88|         42.5|          282|
|                             Bermuda|               90|         22.9|          842|
|        United Kingdom;United States|                9|         20.4|           77|
|          Isle of Man;United Kingdom|               21|         20.2|          339|
|               Bermuda;United States|               21|         18.1|          108|
|                     

The country-level pattern is unambiguous:

- **The most ring-organized footprints** are concentrated in **Bermuda, Jersey, Isle of Man, and the British Virgin Islands** — the classic "trust haven" jurisdictions, with average triangle counts of 20–130 per officer. These are the jurisdictions whose offshore industries are built on tight networks of professional directors and trustees.
- **Multi-country labels with Bermuda or UK** (e.g., "Bermuda;United Kingdom;United States" → 131 avg triangles) represent officers whose names appear across multiple jurisdictions — these are typically the professional fiduciaries who sit on dozens of boards across the offshore world.
- **Large-economy countries** (United States with 621 officers, United Kingdom with 129, Hong Kong with 283) appear in this list but with much lower averages (10–16), suggesting that their offshore footprint is more atomized — many individuals doing offshore activity but without the dense ring structures seen in the trust havens.

The structural finding here is that **offshore professionalism creates rings**. In jurisdictions like Bermuda and Jersey, a small cadre of professional directors sit on each other's boards repeatedly. In larger economies, offshore activity is spread across many uncoordinated individuals. This is exactly the distinction that triangle count was designed to measure, and the projection trick lets us see it clearly in a network that's structurally bipartite.

For a reporter, this would suggest different reporting strategies for different jurisdictions: in Bermuda/Jersey, the story is about the *firms* and *professional rings*; in larger countries, the story is about *individual* prominent figures.